In [1]:
# Install PySpark in Google Colab.
%pip install -q pyspark

In [2]:
# Import the required Python and PySpark libraries.
import os
import shutil
import urllib.request

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType

In [3]:
# Create the required Data Lake directory structure.
BASE_PATH = "/content/data_lake"
BRONZE_PATH = os.path.join(BASE_PATH, "bronze")
SILVER_PATH = os.path.join(BASE_PATH, "silver")
SILVER_OUTPUT_PATH = os.path.join(SILVER_PATH, "preprocessed_sales")

os.makedirs(BRONZE_PATH, exist_ok=True)
os.makedirs(SILVER_PATH, exist_ok=True)

DATASET_URL = (
    "https://raw.githubusercontent.com/Siri-Maya-Lalitha-10"
    "/data-mining-data-lake-assignment/main/dataset/sales.csv"
)

BRONZE_FILE = os.path.join(BRONZE_PATH, "sales.csv")

# Download the original CSV directly into the Bronze layer.
# The Bronze file is not modified after it is downloaded.
urllib.request.urlretrieve(DATASET_URL, BRONZE_FILE)

print("Bronze file:", BRONZE_FILE)
print("Bronze file size (bytes):", os.path.getsize(BRONZE_FILE))

Bronze file: /content/data_lake/bronze/sales.csv
Bronze file size (bytes): 109537


In [4]:
# Create the Spark session required for the assignment.
spark = (
    SparkSession.builder
    .appName("Data Mining Spark Data Lake Assignment")
    .master("local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

Spark version: 4.0.4


In [5]:
# Read the unchanged Bronze CSV with Spark.
# inferSchema is used so Spark can identify numeric columns.
bronze_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("mode", "PERMISSIVE")
    .csv(BRONZE_FILE)
)

# Display the schema, as required.
bronze_df.printSchema()

# Display a few rows, as required.
bronze_df.show(5, truncate=False)

# Store and display the total Bronze record count.
bronze_count = bronze_df.count()
print("Bronze record count:", bronze_count)

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name |product      |category|quantity|unit_price|discount_percent|payment_method|city      |state         |order_date|order_status|
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------

In [6]:
# Identify exact duplicate rows using all columns.
duplicate_groups = (
    bronze_df
    .groupBy(bronze_df.columns)
    .count()
    .filter(F.col("count") > 1)
)

duplicate_group_count = duplicate_groups.count()
duplicate_extra_rows = (
    duplicate_groups
    .select(F.sum(F.col("count") - 1).alias("duplicate_extra_rows"))
    .first()["duplicate_extra_rows"]
)

print("Number of duplicate groups:", duplicate_group_count)
print("Number of extra duplicate rows:", duplicate_extra_rows or 0)

# Display duplicate groups if any exist.
duplicate_groups.show(10, truncate=False)

Number of duplicate groups: 10
Number of extra duplicate rows: 10
+--------+-----------+----------------+-----------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+-----+
|order_id|customer_id|customer_name   |product          |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|count|
+--------+-----------+----------------+-----------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+-----+
|100692  |C0141      |Varun Agarwal   |Electric Kettle  |Home Appliances|4       |920       |0               |Net Banking     |Delhi     |Delhi         |2026-08-18|Shipped     |2    |
|100145  |C0007      |Sai Mishra      |Face Wash        |Beauty         |7       |350       |30              |Cash on Delivery|Bengaluru |Karnataka     |2025-08-28|Delivered   |2    |
|100297  |C014

In [7]:
# Identify null or blank values in every column.
# This is calculated with Spark rather than hard-coded counts.
null_checks = bronze_df.select([
    F.sum(
        F.when(F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""), 1).otherwise(0)
    ).alias(c)
    for c in bronze_df.columns
])

print("Null/blank values by column:")
null_checks.show(truncate=False)

Null/blank values by column:
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|0       |0          |10           |0      |0       |0       |0         |0               |8             |8   |0    |0         |0           |
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+



In [8]:
# Identify text-format inconsistencies in categorical/text fields.
# Distinct values are inspected so obvious variants can be standardized later.
text_columns = [
    "customer_name",
    "product",
    "category",
    "payment_method",
    "city",
    "state",
    "order_status"
]

for column_name in text_columns:
    print(f"\nDistinct values for {column_name}:")
    (
        bronze_df
        .select(F.col(column_name).cast("string").alias(column_name))
        .filter(F.col(column_name).isNotNull())
        .distinct()
        .orderBy(column_name)
        .show(100, truncate=False)
    )


Distinct values for customer_name:
+----------------+
|customer_name   |
+----------------+
|Aarav Agarwal   |
|Aarav Chopra    |
|Aarav Iyer      |
|Aarav Kulkarni  |
|Aarav Kumar     |
|Aarav Nair      |
|Aarav Patel     |
|Aarav Singh     |
|Aarav Tripathi  |
|Abhinav Gupta   |
|Abhinav Kulkarni|
|Abhinav Malhotra|
|Abhinav Pandey  |
|Abhinav Pillai  |
|Abhinav Rao     |
|Abhinav Yadav   |
|Aditi Agarwal   |
|Aditi Chopra    |
|Aditi Kapoor    |
|Aditi Mishra    |
|Aditi Patel     |
|Aditi Tripathi  |
|Aditya Chopra   |
|Aditya Malhotra |
|Aditya Mehta    |
|Aditya Naidu    |
|Aditya Rao      |
|Akash Jain      |
|Akash Jain      |
|Akash Kulkarni  |
|Akash Pillai    |
|Akash Reddy     |
|Akash Saxena    |
|Akash Singh     |
|Akash Yadav     |
|Aman Bose       |
|Aman Chaudhary  |
|Aman Chopra     |
|Aman Das        |
|Aman Jain       |
|Aman Joshi      |
|Aman Kapoor     |
|Aman Naidu      |
|Aman Nair       |
|Aman Rao        |
|Aman Saxena     |
|Aman Shetty     |
|Aman Tripathi

In [9]:
# Identify invalid numerical values.
# Quantity must be greater than zero.
# Unit price must be greater than zero.
# Discount percentage must be between 0 and 100 inclusive.

numeric_issues = bronze_df.select(
    F.sum(F.when(F.col("quantity") <= 0, 1).otherwise(0)).alias("invalid_quantity_rows"),
    F.sum(F.when(F.col("unit_price") <= 0, 1).otherwise(0)).alias("invalid_unit_price_rows"),
    F.sum(
        F.when(
            (F.col("discount_percent") < 0) | (F.col("discount_percent") > 100),
            1
        ).otherwise(0)
    ).alias("invalid_discount_rows")
)

print("Invalid numerical values:")
numeric_issues.show()

Invalid numerical values:
+---------------------+-----------------------+---------------------+
|invalid_quantity_rows|invalid_unit_price_rows|invalid_discount_rows|
+---------------------+-----------------------+---------------------+
|                   12|                      8|                    8|
+---------------------+-----------------------+---------------------+



In [10]:
# Identify invalid or inconsistent dates.
# try_to_timestamp safely returns NULL for values that cannot be parsed.

date_check_df = (
    bronze_df
    .withColumn(
        "parsed_order_date",
        F.expr("""
            try_to_timestamp(
                trim(cast(order_date as string)),
                'yyyy-MM-dd'
            )
        """)
    )
)

date_check = (
    date_check_df
    .select(
        F.sum(
            F.when(
                F.col("order_date").isNull()
                | (F.trim(F.col("order_date").cast("string")) == ""),
                1
            ).otherwise(0)
        ).alias("missing_order_date_rows"),

        F.sum(
            F.when(
                F.col("order_date").isNotNull()
                & (F.trim(F.col("order_date").cast("string")) != "")
                & F.col("parsed_order_date").isNull(),
                1
            ).otherwise(0)
        ).alias("invalid_order_date_rows")
    )
)

print("Date validation:")
date_check.show()

# Display invalid date values so they can be inspected.
(
    date_check_df
    .filter(
        F.col("order_date").isNotNull()
        & (F.trim(F.col("order_date").cast("string")) != "")
        & F.col("parsed_order_date").isNull()
    )
    .select("order_date")
    .distinct()
    .show(20, truncate=False)
)

Date validation:
+-----------------------+-----------------------+
|missing_order_date_rows|invalid_order_date_rows|
+-----------------------+-----------------------+
|                      0|                      6|
+-----------------------+-----------------------+

+----------+
|order_date|
+----------+
|2025/07/14|
|31/02/2026|
|not-a-date|
|2025-02-30|
|2026-13-05|
|09-15-2026|
+----------+



In [11]:
# Remove exact duplicate rows as the first Silver transformation.
silver_df = bronze_df.dropDuplicates()

# Trim whitespace from all text-like columns.
# Names and product names are trimmed without forcing artificial capitalization changes.
for column_name in text_columns:
    silver_df = silver_df.withColumn(
        column_name,
        F.when(
            F.col(column_name).isNull(),
            F.lit(None)
        ).otherwise(F.trim(F.col(column_name).cast("string")))
    )

In [12]:
# Handle missing text values with an explicit, neutral placeholder.
# This preserves otherwise useful records instead of dropping them.
for column_name in text_columns:
    silver_df = silver_df.withColumn(
        column_name,
        F.when(
            F.col(column_name).isNull() | (F.col(column_name) == ""),
            F.lit("Unknown")
        ).otherwise(F.col(column_name))
    )

# Standardize obvious capitalization inconsistencies in categorical fields.
# Known category variants are normalized without changing the meaning of records.
silver_df = silver_df.withColumn(
    "category",
    F.initcap(F.lower(F.col("category")))
)

# Normalize payment methods to a consistent set of labels.
payment_method_normalized = F.lower(F.trim(F.col("payment_method")))
silver_df = silver_df.withColumn(
    "payment_method",
    F.when(payment_method_normalized == "credit card", "Credit Card")
     .when(payment_method_normalized == "debit card", "Debit Card")
     .when(payment_method_normalized == "cash on delivery", "Cash on Delivery")
     .when(payment_method_normalized == "net banking", "Net Banking")
     .when(payment_method_normalized == "upi", "UPI")
     .otherwise(F.col("payment_method"))
)

# Standardize city, state, and order status capitalization.
silver_df = silver_df.withColumn("city", F.initcap(F.lower(F.col("city"))))
silver_df = silver_df.withColumn("state", F.initcap(F.lower(F.col("state"))))
silver_df = silver_df.withColumn("order_status", F.initcap(F.lower(F.col("order_status"))))

In [13]:
# Convert numerical columns to appropriate numeric types.
silver_df = (
    silver_df
    .withColumn("quantity", F.col("quantity").cast(IntegerType()))
    .withColumn("unit_price", F.col("unit_price").cast(DoubleType()))
    .withColumn("discount_percent", F.col("discount_percent").cast(DoubleType()))
)

# Mark clearly invalid numerical values as NULL.
# This keeps the transaction record while removing invalid numeric measurements.
silver_df = (
    silver_df
    .withColumn(
        "quantity",
        F.when(F.col("quantity") > 0, F.col("quantity")).otherwise(F.lit(None).cast("int"))
    )
    .withColumn(
        "unit_price",
        F.when(F.col("unit_price") > 0, F.col("unit_price")).otherwise(F.lit(None).cast("double"))
    )
    .withColumn(
        "discount_percent",
        F.when(
            (F.col("discount_percent") >= 0) & (F.col("discount_percent") <= 100),
            F.col("discount_percent")
        ).otherwise(F.lit(None).cast("double"))
    )
)

In [14]:
# Handle missing/invalid numeric values using median imputation.
# Median is robust to unusually large or small valid values.
numeric_columns = ["quantity", "unit_price", "discount_percent"]

median_values = {}
for column_name in numeric_columns:
    median_values[column_name] = silver_df.approxQuantile(
        column_name, [0.5], 0.001
    )[0]

print("Median values used for numeric imputation:")
for column_name, median_value in median_values.items():
    print(column_name, "=", median_value)

for column_name in numeric_columns:
    silver_df = silver_df.fillna({column_name: median_values[column_name]})

Median values used for numeric imputation:
quantity = 5.0
unit_price = 1840.0
discount_percent = 20.0


In [15]:
# Convert valid dates to one consistent YYYY-MM-DD format.
# Invalid dates are converted to NULL while preserving the record.

silver_df = silver_df.withColumn(
    "order_date",
    F.date_format(
        F.expr("""
            try_to_timestamp(
                trim(cast(order_date as string)),
                'yyyy-MM-dd'
            )
        """),
        "yyyy-MM-dd"
    )
)

In [16]:
# Reorder columns to preserve the original dataset structure.
original_column_order = bronze_df.columns
silver_df = silver_df.select(*original_column_order)

# Display a sample of the preprocessed Silver DataFrame.
silver_df.show(10, truncate=False)

# Display the final Silver schema.
silver_df.printSchema()

+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name     |product               |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100331  |C0005      |Sakshi Joshi      |Football              |Sports         |2       |670.0     |30.0            |Credit Card     |Bengaluru |Karnataka     |2025-06-06|Cancelled   |
|100333  |C0263      |Siddharth Kulkarni|Programming Book      |Books          |4       |670.0     |0.0             |Net Banking     |Pune      |Maharashtra   |2025-05-08|Delivered   |
|100506  |C0228      |Diya Tripathi     |Competitive Exam Guide|Books      

In [17]:
# Remove an existing Silver output directory so the notebook can be rerun cleanly.
if os.path.exists(SILVER_OUTPUT_PATH):
    shutil.rmtree(SILVER_OUTPUT_PATH)

# Save the Silver layer as a single CSV directory.
# Spark normally creates multiple part files; coalesce(1) creates one CSV part file
# for convenient submission as sales_silver.csv.
(
    silver_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(SILVER_OUTPUT_PATH)
)

print("Silver output directory:", SILVER_OUTPUT_PATH)

Silver output directory: /content/data_lake/silver/preprocessed_sales


In [18]:
# Rename Spark's generated CSV part file to sales_silver.csv.
silver_part_files = [
    filename
    for filename in os.listdir(SILVER_OUTPUT_PATH)
    if filename.startswith("part-") and filename.endswith(".csv")
]

if len(silver_part_files) != 1:
    raise RuntimeError(
        f"Expected exactly one Silver CSV part file, found {len(silver_part_files)}."
    )

silver_part_file = os.path.join(SILVER_OUTPUT_PATH, silver_part_files[0])
silver_csv_file = os.path.join(SILVER_PATH, "sales_silver.csv")

if os.path.exists(silver_csv_file):
    os.remove(silver_csv_file)

os.rename(silver_part_file, silver_csv_file)

# Remove Spark metadata files from the final Silver submission directory.
for filename in os.listdir(SILVER_OUTPUT_PATH):
    filepath = os.path.join(SILVER_OUTPUT_PATH, filename)
    if os.path.isfile(filepath):
        os.remove(filepath)

os.rmdir(SILVER_OUTPUT_PATH)

print("Silver CSV:", silver_csv_file)

Silver CSV: /content/data_lake/silver/sales_silver.csv


In [19]:
# Verification: show row counts before and after preprocessing.
silver_count = silver_df.count()

print("Row count before preprocessing:", bronze_count)
print("Row count after preprocessing:", silver_count)
print("Rows removed by duplicate removal:", bronze_count - silver_count)

Row count before preprocessing: 1000
Row count after preprocessing: 990
Rows removed by duplicate removal: 10


In [20]:
# Read the Silver CSV back with Spark to verify that the saved output is readable.
verified_silver_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(silver_csv_file)
)

print("Silver row count after reading from disk:", verified_silver_df.count())

# Display a few rows from the saved Silver data.
verified_silver_df.show(10, truncate=False)

# Display the saved Silver schema.
verified_silver_df.printSchema()

Silver row count after reading from disk: 990
+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name     |product               |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100331  |C0005      |Sakshi Joshi      |Football              |Sports         |2       |670.0     |30.0            |Credit Card     |Bengaluru |Karnataka     |2025-06-06|Cancelled   |
|100333  |C0263      |Siddharth Kulkarni|Programming Book      |Books          |4       |670.0     |0.0             |Net Banking     |Pune      |Maharashtra   |2025-05-08|Delivered   |
|100506  |C0228      |Diya Tr

In [21]:
# Verify that the Silver output no longer contains null/blank text fields
# for the text columns handled above.
silver_text_nulls = verified_silver_df.select([
    F.sum(
        F.when(
            F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""),
            1
        ).otherwise(0)
    ).alias(c)
    for c in text_columns
])

print("Remaining null/blank text values in Silver:")
silver_text_nulls.show(truncate=False)

Remaining null/blank text values in Silver:
+-------------+-------+--------+--------------+----+-----+------------+
|customer_name|product|category|payment_method|city|state|order_status|
+-------------+-------+--------+--------------+----+-----+------------+
|0            |0      |0       |0             |0   |0    |0           |
+-------------+-------+--------+--------------+----+-----+------------+



In [22]:
# Verify numerical validation in the saved Silver data.
silver_numeric_issues = verified_silver_df.select(
    F.sum(F.when(F.col("quantity") <= 0, 1).otherwise(0)).alias("invalid_quantity_rows"),
    F.sum(F.when(F.col("unit_price") <= 0, 1).otherwise(0)).alias("invalid_unit_price_rows"),
    F.sum(
        F.when(
            (F.col("discount_percent") < 0) | (F.col("discount_percent") > 100),
            1
        ).otherwise(0)
    ).alias("invalid_discount_rows")
)

print("Invalid numerical values remaining in Silver:")
silver_numeric_issues.show()

Invalid numerical values remaining in Silver:
+---------------------+-----------------------+---------------------+
|invalid_quantity_rows|invalid_unit_price_rows|invalid_discount_rows|
+---------------------+-----------------------+---------------------+
|                    0|                      0|                    0|
+---------------------+-----------------------+---------------------+



In [23]:
# Verify that valid Silver dates use the required YYYY-MM-DD representation.
# Invalid source dates remain NULL by design.
silver_date_validation = (
    verified_silver_df
    .select(
        F.sum(
            F.when(
                F.col("order_date").isNotNull()
                & F.to_date(F.col("order_date"), "yyyy-MM-dd").isNull(),
                1
            ).otherwise(0)
        ).alias("invalid_silver_dates")
    )
)

print("Invalid non-null dates remaining in Silver:")
silver_date_validation.show()

Invalid non-null dates remaining in Silver:
+--------------------+
|invalid_silver_dates|
+--------------------+
|                   0|
+--------------------+



In [24]:
# Print the final Silver file path for use during GitHub submission.
print("Final files created:")
print("Bronze:", BRONZE_FILE)
print("Silver:", silver_csv_file)

# Stop the Spark session after all verification is complete.
spark.stop()

Final files created:
Bronze: /content/data_lake/bronze/sales.csv
Silver: /content/data_lake/silver/sales_silver.csv
